# Questão 3 — Custos de Importação

## Contexto

O arquivo `custos_importacao.json` armazena o histórico de preços de compra de cada produto em um campo aninhado chamado `historic_data`. Isso significa que cada produto tem vários registros de preço dentro de uma lista, e precisamos "achatar" essa estrutura para uma tabela plana — uma linha por entrada de preço.

### Estrutura original (JSON aninhado)

```json
{
  "product_id": 1,
  "product_name": "Transponder AIS Maré Magnum",
  "category": "eletrônicos",
  "historic_data": [
    { "start_date": "10/08/2016", "usd_price": 10583.63 },
    { "start_date": "15/06/2018", "usd_price": 8778.36 },
    ...
  ]
}
```

### Estrutura desejada (CSV plano)

| product_id | product_name | category | start_date | usd_price |
|---|---|---|---|---|
| 1 | Transponder AIS Maré Magnum | eletrônicos | 2016-08-10 | 10583.63 |
| 1 | Transponder AIS Maré Magnum | eletrônicos | 2018-06-15 | 8778.36 |
| ... | ... | ... | ... | ... |

---

In [ ]:
import json
import pandas as pd

with open('../data/raw/custos_importacao.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

print(f'Total de produtos no JSON: {len(dados)}') # Quantos produtos existem?

print('\nExemplo do primeiro produto:')
print(json.dumps(dados[0], indent=2, ensure_ascii=False))

Total de produtos no JSON: 150

Exemplo do primeiro produto:
{
  "product_id": 1,
  "product_name": "Transponder AIS Maré Magnum",
  "category": "eletrônicos",
  "historic_data": [
    {
      "start_date": "10/08/2016",
      "usd_price": 10583.63
    },
    {
      "start_date": "15/06/2018",
      "usd_price": 8778.36
    },
    {
      "start_date": "25/09/2018",
      "usd_price": 8023.87
    },
    {
      "start_date": "19/03/2019",
      "usd_price": 8772.78
    },
    {
      "start_date": "17/01/2020",
      "usd_price": 7918.18
    },
    {
      "start_date": "17/06/2020",
      "usd_price": 6310.01
    },
    {
      "start_date": "02/07/2021",
      "usd_price": 6586.7
    },
    {
      "start_date": "16/05/2022",
      "usd_price": 6538.2
    },
    {
      "start_date": "28/02/2023",
      "usd_price": 6360.91
    },
    {
      "start_date": "17/10/2023",
      "usd_price": 6574.8
    },
    {
      "start_date": "16/02/2024",
      "usd_price": 6657.12
    },
    {
 

Cada produto pode ter vários registros históricos de preço dentro da lista `historic_data`.
Precisamos transformar isso em uma tabela onde cada linha representa um preço em uma data específica.

Vamos ver quantos registros históricos o primeiro produto tem:

In [2]:
primeiro_produto = dados[0]

print(f'Produto: {primeiro_produto["product_name"]}')
print(f'Quantidade de registros históricos: {len(primeiro_produto["historic_data"])}')
print()

# Mostra cada entrada histórica
for entrada in primeiro_produto['historic_data']:
    print(f'  Data: {entrada["start_date"]}  |  Preço USD: {entrada["usd_price"]}')

Produto: Transponder AIS Maré Magnum
Quantidade de registros históricos: 15

  Data: 10/08/2016  |  Preço USD: 10583.63
  Data: 15/06/2018  |  Preço USD: 8778.36
  Data: 25/09/2018  |  Preço USD: 8023.87
  Data: 19/03/2019  |  Preço USD: 8772.78
  Data: 17/01/2020  |  Preço USD: 7918.18
  Data: 17/06/2020  |  Preço USD: 6310.01
  Data: 02/07/2021  |  Preço USD: 6586.7
  Data: 16/05/2022  |  Preço USD: 6538.2
  Data: 28/02/2023  |  Preço USD: 6360.91
  Data: 17/10/2023  |  Preço USD: 6574.8
  Data: 16/02/2024  |  Preço USD: 6657.12
  Data: 22/02/2024  |  Preço USD: 6703.2
  Data: 15/03/2024  |  Preço USD: 6633.66
  Data: 02/08/2024  |  Preço USD: 5774.5
  Data: 08/04/2025  |  Preço USD: 5579.75


Normalizando o JSON em uma tabela plana

In [3]:
# pd.json_normalize expande a lista aninhada:
# - 'record_path' indica qual campo é a lista a expandir
# - 'meta' indica quais campos do nível superior repetir em cada linha
df = pd.json_normalize(
    dados,
    record_path='historic_data',
    meta=['product_id', 'product_name', 'category']
)

print(f'Shape após normalização: {df.shape}')
print()
print('Colunas geradas:', df.columns.tolist())
print()
df.head()

Shape após normalização: (1260, 5)

Colunas geradas: ['start_date', 'usd_price', 'product_id', 'product_name', 'category']



,start_date,usd_price,product_id,product_name,category
0,10/08/2016,10583.63,1,Transponder AIS Maré Magnum,eletrônicos
1,15/06/2018,8778.36,1,Transponder AIS Maré Magnum,eletrônicos
2,25/09/2018,8023.87,1,Transponder AIS Maré Magnum,eletrônicos
3,19/03/2019,8772.78,1,Transponder AIS Maré Magnum,eletrônicos
4,17/01/2020,7918.18,1,Transponder AIS Maré Magnum,eletrônicos


Colunas fora da ordem esperada. Reordenando para ficar igual a definição do db:

`product_id` → `product_name` → `category` → `start_date` → `usd_price`

In [4]:
df = df[['product_id', 'product_name', 'category', 'start_date', 'usd_price']]

df['product_id'] = df['product_id'].astype(int)
df['usd_price']  = df['usd_price'].astype(float)

# Converte start_date de DD/MM/YYYY para o formato date padrão YYYY-MM-DD
df['start_date'] = pd.to_datetime(df['start_date'], format='%d/%m/%Y').dt.date

print('Tipos das colunas:')
print(df.dtypes)
print()
df.head(10)

Tipos das colunas:
product_id        int64
product_name        str
category            str
start_date       object
usd_price       float64
dtype: object



,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,2016-08-10,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,2018-06-15,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,2018-09-25,8023.87
3,1,Transponder AIS Maré Magnum,eletrônicos,2019-03-19,8772.78
4,1,Transponder AIS Maré Magnum,eletrônicos,2020-01-17,7918.18
5,1,Transponder AIS Maré Magnum,eletrônicos,2020-06-17,6310.01
6,1,Transponder AIS Maré Magnum,eletrônicos,2021-07-02,6586.70
7,1,Transponder AIS Maré Magnum,eletrônicos,2022-05-16,6538.20
8,1,Transponder AIS Maré Magnum,eletrônicos,2023-02-28,6360.91
9,1,Transponder AIS Maré Magnum,eletrônicos,2023-10-17,6574.80


## Validando o resultado
- Quantidade total de linhas
- Ausência de nulos
- Distribuição por categoria

In [5]:
print(f'Total de entradas no CSV: {len(df)}')
print()
print('Valores nulos por coluna:')
print(df.isnull().sum())
print()
print('Registros por categoria:')
print(df['category'].value_counts())
print()
print(f'Intervalo de datas: {df["start_date"].min()} → {df["start_date"].max()}')

Total de entradas no CSV: 1260

Valores nulos por coluna:
product_id      0
product_name    0
category        0
start_date      0
usd_price       0
dtype: int64

Registros por categoria:
category
eletrônicos    436
ancoragem      413
propulsão      411
Name: count, dtype: int64

Intervalo de datas: 2016-01-04 → 2025-12-31


Export CSV

In [6]:
import os

os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/custos_importacao_normalizado.csv', index=False, encoding='utf-8')

print('Arquivo salvo: custos_importacao_normalizado.csv')
print(f'Total de linhas exportadas: {len(df)}')

Arquivo salvo: custos_importacao_normalizado.csv
Total de linhas exportadas: 1260
